# Coffee17 — Preprocessing Nuisance Analysis\n\nTujuan: menguji apakah R0/C0/F0/W0 mempertahankan representasi defect di bawah gangguan acquisition-like yang dibekukan. Ini **bukan** uji camera invariance fisik.\n\nNotebook menggunakan checkpoint frozen MVFD-SBN. Jika output project dari notebook MVFD sebelumnya dilampirkan, checkpoint akan digunakan langsung; jika tidak, fold yang hilang direproduksi dengan scientific commit frozen.\n\n**Sebelum Run All:** attach Coffee17 original, aktifkan GPU + Internet. Opsional: attach saved Kaggle output yang mengandung folder `coffee17-mvfd-sbn-project` agar tidak perlu training ulang.\n

In [ ]:
# Coffee17 preprocessing nuisance analysis
ANALYSIS_COMMIT = "62f7f8b3755d725381d2c265f3387acb3be70afc"
FROZEN_MVFD_COMMIT = "fdae1fff10348f78ca67bb00948ae5768e9c98d9"

import hashlib, importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-nuisance-code"
MVFD_PROJECT = WORK / "coffee17-mvfd-sbn-project"
PROJECT = WORK / "coffee17-preprocessing-nuisance-project"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook harus dijalankan di Kaggle."

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def merge_tree_exact(source, target):
    source, target = Path(source), Path(target)
    for item in sorted(source.rglob("*")):
        if not item.is_file():
            continue
        rel = item.relative_to(source)
        dst = target / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.is_file():
            if sha256_file(item) != sha256_file(dst):
                raise RuntimeError(f"Conflict saat resume: {rel}")
        else:
            shutil.copy2(item, dst)

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# Reuse frozen MVFD checkpoints if the saved project from the earlier notebook
# is attached as a Kaggle Input. Otherwise missing folds are reproduced.
for prior in sorted(p for p in INPUT.rglob("coffee17-mvfd-sbn-project") if p.is_dir()):
    print("FOUND PRIOR MVFD PROJECT:", prior)
    merge_tree_exact(prior, MVFD_PROJECT)

for prior in sorted(p for p in INPUT.rglob("coffee17-preprocessing-nuisance-project") if p.is_dir()):
    print("RESUME NUISANCE PROJECT:", prior)
    merge_tree_exact(prior, PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
ANALYSIS_OUT = PROJECT / "analysis"
ANALYSIS_OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Checkout analysis code and install dependencies.
# ------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", ANALYSIS_COMMIT])
run([sys.executable, "-m", "pip", "install", "-q", "-r", REPO / "requirements/preprocessing-study.txt"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import materialize_preprocessing_development
from bilinear_lmmd.engine.shared_multiview import validation_identity_label_sha256

REFERENCE = REPO / "configs/shared_multiview/sbn_reference_v1.json"
CONFIG = REPO / "configs/mvfd_sbn/MVFD_SBN_ALL4.yaml"

# ------------------------------------------------------------
# 2. Reconstruct exact Coffee17 development folds.
# ------------------------------------------------------------
print("\n=== 1/4 RECONSTRUCT COFFEE17 ===")
by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus memuat Coffee17 original valid saja "
        f"(expected 979/17, observed {raw_count}/{len(by_class)})."
    )

ARCHIVE = WORK / "coffee17_nuisance_original.zip"
PROV_LOCAL = WORK / "coffee17_nuisance_provenance"
CANONICAL = WORK / "coffee17_nuisance_original_v1"
FOLDS_LOCAL = WORK / "coffee17_nuisance_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE, PROV_LOCAL, canonical_root=CANONICAL
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

reference = json.loads(REFERENCE.read_text(encoding="utf-8"))

# ------------------------------------------------------------
# 3. Ensure frozen MVFD checkpoints exist, then diagnose nuisances.
# ------------------------------------------------------------
print("\n=== 2/4 FROZEN CHECKPOINTS + NUISANCE DIAGNOSTIC ===")
MVFD_OUT = MVFD_PROJECT / "experiments/coffee17-mvfd-sbn-v1"
MVFD_OUT.mkdir(parents=True, exist_ok=True)

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_nuisance_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)
    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    count, val_sha = validation_identity_label_sha256(DEV)
    expected = reference["folds"][f"fold_{fold}"]
    if count != expected["count"] or val_sha != expected["identity_label_sha256"]:
        raise RuntimeError(f"Fold {fold}: validation authority mismatch.")
    print("VALIDATION REFERENCE: PASS", count, val_sha)

    run_dir = MVFD_OUT / "MVFD_SBN_ALL4" / f"fold_{fold}" / "seed42"
    checkpoint = run_dir / "best.pt"

    if not checkpoint.is_file():
        print("Frozen MVFD checkpoint missing; reproducing this fold.")
        run(["git", "-C", REPO, "checkout", "--quiet", "--detach", FROZEN_MVFD_COMMIT])
        run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
        run([
            sys.executable, "-u", "-m",
            "bilinear_lmmd.experiments.run_mvfd_sbn",
            "--config", REPO / "configs/mvfd_sbn/MVFD_SBN_ALL4.yaml",
            "--reference", REPO / "configs/shared_multiview/sbn_reference_v1.json",
            "--fold", str(fold),
            "--data-root", DEV,
            "--output-root", MVFD_OUT,
            "--required-commit", FROZEN_MVFD_COMMIT,
            "--device", "cuda:0",
            "--resume",
            "--authorize-training",
        ], cwd=REPO, log_path=PROJECT / "logs" / f"reproduce_mvfd_fold{fold}.log")

    if not checkpoint.is_file():
        raise RuntimeError(f"Fold {fold}: best.pt tidak tersedia.")

    # Restore the analysis checkout before running the diagnostic.
    run(["git", "-C", REPO, "checkout", "--quiet", "--detach", ANALYSIS_COMMIT])
    run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])

    fold_json = ANALYSIS_OUT / f"fold_{fold}.json"
    if not fold_json.is_file():
        run([
            sys.executable, "-u", "-m",
            "bilinear_lmmd.experiments.run_mvfd_nuisance_analysis",
            "--checkpoint", checkpoint,
            "--data-root", DEV,
            "--output", fold_json,
            "--fold", str(fold),
            "--device", "cuda:0",
        ], cwd=REPO, log_path=PROJECT / "logs" / f"nuisance_fold{fold}.log")

    result = json.loads(fold_json.read_text(encoding="utf-8"))
    for nuisance in result["nuisance_kinds"]:
        row = result["nuisances"][nuisance]
        print(
            f"{nuisance:20s} | "
            f"cos R0={row['R0']['cosine_stability']:.4f} "
            f"F0={row['F0']['cosine_stability']:.4f} "
            f"W0={row['W0']['cosine_stability']:.4f} | "
            f"Macro drop R0={row['R0']['macro_f1_drop']:+.4f} "
            f"F0={row['F0']['macro_f1_drop']:+.4f} "
            f"W0={row['W0']['macro_f1_drop']:+.4f}"
        )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Summary and compact package.
# ------------------------------------------------------------
print("\n=== 3/4 SUMMARY ===")
SUMMARY = ANALYSIS_OUT / "preprocessing_nuisance_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_mvfd_nuisance_summary",
    "--input-dir", ANALYSIS_OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
print("\n=== DESCRIPTIVE ROBUSTNESS COUNT ===")
for arm, value in summary["descriptive_robustness_count"].items():
    print(arm, value["qualified_nuisances"], "/", value["out_of"])

print("\n=== 4/4 BUILD PACKAGE ===")
PKG = WORK / "preprocessing-nuisance-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)
shutil.copytree(ANALYSIS_OUT, PKG / "analysis")
shutil.copy2(
    REPO / "docs/protocols/PREPROCESSING_NUISANCE_ANALYSIS_V1.md",
    PKG / "PREPROCESSING_NUISANCE_ANALYSIS_V1.md",
)

manifest = {
    "analysis_commit": ANALYSIS_COMMIT,
    "frozen_mvfd_commit": FROZEN_MVFD_COMMIT,
    "analysis_type": "synthetic acquisition-nuisance diagnostic",
    "views": ["R0", "C0", "F0", "W0"],
    "folds": 5,
    "validation_images_per_fold": 97,
    "outer_test_accessed": False,
    "claim_boundary": (
        "This analysis can support perturbation-specific robustness claims, "
        "not physical camera/illumination invariance."
    ),
}
(PKG / "manifest.json").write_text(
    json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
)

zip_path = Path(shutil.make_archive(
    str(WORK / "preprocessing-nuisance-analysis-package"),
    "zip",
    root_dir=PKG,
))
print("\nREADY:", zip_path)
print("SIZE :", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")

for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
